# R Integration Example - Advanced Statistical Analysis

This notebook demonstrates how to use R integration features in the menstrual_cycle_analysis package for advanced statistical analyses.

**Note**: This notebook requires `rpy2` to be installed. Install with: `pip install rpy2`

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

# Import our analysis package with R integration
from menstrual_cycle_analysis import DataProcessor, SleepAnalyzer, CycleAnalyzer

# Check if R integration is available
try:
    import rpy2.robjects as ro
    print("R integration available!")
    R_AVAILABLE = True
except ImportError:
    print("R integration not available. Install rpy2 to use R features.")
    R_AVAILABLE = False

## 1. Create Sample Data with More Complex Patterns

In [ ]:
# Create more realistic sample data with cycle-related patterns
np.random.seed(42)

# Generate 120 days of data (about 4 cycles)
dates = pd.date_range(start='2024-01-01', periods=120, freq='D')

# Create cycle phase pattern (28-day cycles)
cycle_day = [(i % 28) + 1 for i in range(120)]

# Add cycle-related variations to sleep and physiological data
sleep_duration_base = 7.5
sleep_efficiency_base = 85
heart_rate_base = 65
temperature_base = 98.6

# Simulate cycle effects
sleep_variations = []
efficiency_variations = []
hr_variations = []
temp_variations = []

for day in cycle_day:
    if day <= 5:  # Menstrual phase
        sleep_variations.append(np.random.normal(-0.3, 0.2))
        efficiency_variations.append(np.random.normal(-3, 2))
        hr_variations.append(np.random.normal(2, 1))
        temp_variations.append(np.random.normal(-0.1, 0.1))
    elif day <= 13:  # Follicular phase
        sleep_variations.append(np.random.normal(0.1, 0.2))
        efficiency_variations.append(np.random.normal(1, 2))
        hr_variations.append(np.random.normal(-1, 1))
        temp_variations.append(np.random.normal(0, 0.1))
    elif day <= 15:  # Ovulatory phase
        sleep_variations.append(np.random.normal(-0.2, 0.3))
        efficiency_variations.append(np.random.normal(-2, 3))
        hr_variations.append(np.random.normal(3, 2))
        temp_variations.append(np.random.normal(0.3, 0.1))
    else:  # Luteal phase
        sleep_variations.append(np.random.normal(-0.1, 0.2))
        efficiency_variations.append(np.random.normal(-1, 2))
        hr_variations.append(np.random.normal(1, 1))
        temp_variations.append(np.random.normal(0.2, 0.1))

sample_data = pd.DataFrame({
    'date': dates,
    'cycle_day': cycle_day,
    'sleep_duration': sleep_duration_base + np.array(sleep_variations) + np.random.normal(0, 0.5, 120),
    'sleep_efficiency': sleep_efficiency_base + np.array(efficiency_variations) + np.random.normal(0, 5, 120),
    'rem_sleep': np.random.normal(1.8, 0.3, 120),
    'deep_sleep': np.random.normal(1.2, 0.2, 120),
    'heart_rate': heart_rate_base + np.array(hr_variations) + np.random.normal(0, 3, 120),
    'body_temperature': temperature_base + np.array(temp_variations) + np.random.normal(0, 0.2, 120),
    'stress_level': np.random.normal(5, 2, 120),  # 1-10 scale
    'exercise_duration': np.random.exponential(30, 120)  # minutes
})

# Ensure realistic ranges
sample_data['sleep_duration'] = np.clip(sample_data['sleep_duration'], 4, 12)
sample_data['sleep_efficiency'] = np.clip(sample_data['sleep_efficiency'], 60, 100)
sample_data['stress_level'] = np.clip(sample_data['stress_level'], 1, 10)
sample_data['exercise_duration'] = np.clip(sample_data['exercise_duration'], 0, 120)

print(f"Created enhanced sample dataset with {len(sample_data)} days of data")
sample_data.head()

## 2. Initialize Analyzers with R Integration

In [ ]:
if R_AVAILABLE:
    # Initialize with R integration enabled
    processor = DataProcessor(use_r=True)
    sleep_analyzer = SleepAnalyzer(use_r=True)
    cycle_analyzer = CycleAnalyzer(use_r=True)
    
    print("Analyzers initialized with R integration")
else:
    # Fallback to Python-only
    processor = DataProcessor(use_r=False)
    sleep_analyzer = SleepAnalyzer(use_r=False)
    cycle_analyzer = CycleAnalyzer(use_r=False)
    
    print("Analyzers initialized without R integration")

# Load data
processor.data = sample_data.copy()
sleep_analyzer.load_sleep_data(sample_data)
cycle_analyzer.load_cycle_data(sample_data)

## 3. Advanced R Statistical Analysis

In [ ]:
if R_AVAILABLE:
    # Example 1: Custom R analysis for sleep patterns
    r_sleep_analysis = """
    # Load required R packages
    if (!require(stats)) install.packages("stats")
    
    # Convert cycle_day to factor for proper analysis
    data$cycle_day_factor <- as.factor(data$cycle_day)
    
    # Perform linear regression
    sleep_model <- lm(sleep_duration ~ cycle_day + heart_rate + stress_level, data=data)
    
    # Get model summary
    model_summary <- summary(sleep_model)
    
    # Return key statistics
    list(
        r_squared = model_summary$r.squared,
        adj_r_squared = model_summary$adj.r.squared,
        f_statistic = model_summary$fstatistic[1],
        p_value = pf(model_summary$fstatistic[1], 
                    model_summary$fstatistic[2], 
                    model_summary$fstatistic[3], 
                    lower.tail=FALSE),
        coefficients = coef(sleep_model)
    )
    """
    
    try:
        result = processor.apply_r_function(r_sleep_analysis)
        print("R Sleep Analysis Results:")
        print(f"R-squared: {float(result[0][0]):.4f}")
        print(f"Adjusted R-squared: {float(result[1][0]):.4f}")
        print(f"F-statistic: {float(result[2][0]):.4f}")
        print(f"P-value: {float(result[3][0]):.6f}")
    except Exception as e:
        print(f"R analysis failed: {e}")
else:
    print("R integration not available - skipping R analysis")

## 4. Advanced Cycle Phase Analysis with R

In [ ]:
# Define menstruation dates for phase identification
menstruation_dates = [
    datetime(2024, 1, 1),
    datetime(2024, 1, 29),
    datetime(2024, 2, 26),
    datetime(2024, 3, 25),
    datetime(2024, 4, 22)
]

# Identify cycle phases
cycle_phases = cycle_analyzer.identify_cycle_phases(menstruation_dates)

if R_AVAILABLE:
    # Custom R script for advanced cycle analysis
    r_cycle_analysis = """
    # Add cycle phases to the data
    phases <- c(""" + '", "'.join([str(p) for p in cycle_phases.values]) + """)
    cycle_data$phase <- factor(phases)
    
    # Remove any rows with missing phase data
    clean_data <- cycle_data[complete.cases(cycle_data$phase), ]
    
    # Perform MANOVA (Multivariate ANOVA)
    if (length(unique(clean_data$phase)) > 1) {
        # Create dependent variable matrix
        Y <- cbind(clean_data$sleep_duration, 
                   clean_data$sleep_efficiency, 
                   clean_data$heart_rate)
        
        # Perform MANOVA
        manova_result <- manova(Y ~ phase, data = clean_data)
        manova_summary <- summary(manova_result)
        
        # Get Pillai's trace statistic
        pillai <- manova_summary$stats[1, "Pillai"]
        
        list(
            test = "MANOVA",
            pillai_trace = pillai,
            phases_analyzed = length(unique(clean_data$phase)),
            sample_size = nrow(clean_data)
        )
    } else {
        list(error = "Insufficient phase data for MANOVA")
    }
    """
    
    try:
        result = cycle_analyzer.run_r_analysis(r_cycle_analysis)
        if result['success']:
            r_result = result['result']
            print("R Cycle Analysis Results:")
            if 'error' not in r_result.names:
                print(f"Test: {r_result[0][0]}")
                print(f"Pillai's Trace: {float(r_result[1][0]):.4f}")
                print(f"Phases analyzed: {int(r_result[2][0])}")
                print(f"Sample size: {int(r_result[3][0])}")
            else:
                print(f"Error: {r_result[0][0]}")
        else:
            print(f"R analysis failed: {result['error']}")
    except Exception as e:
        print(f"R cycle analysis failed: {e}")
else:
    print("R integration not available - using Python statistical analysis")
    
    # Fallback to Python analysis
    stats_result = sleep_analyzer.compare_phases_statistical(
        cycle_phases, 
        sleep_metric='sleep_duration'
    )
    print("Python Statistical Analysis:")
    print(f"Test: {stats_result['test']}")
    if 'f_statistic' in stats_result:
        print(f"F-statistic: {stats_result['f_statistic']:.4f}")
        print(f"P-value: {stats_result['p_value']:.4f}")

## 5. Time Series Analysis with R

In [ ]:
if R_AVAILABLE:
    # Time series decomposition using R
    r_timeseries = """
    # Create time series object for sleep duration
    sleep_ts <- ts(cycle_data$sleep_duration, frequency = 28)  # 28-day cycle
    
    # Decompose the time series
    if (length(sleep_ts) >= 56) {  # Need at least 2 cycles
        decomp <- decompose(sleep_ts, type = "additive")
        
        # Calculate variance explained by components
        trend_var <- var(decomp$trend, na.rm = TRUE)
        seasonal_var <- var(decomp$seasonal, na.rm = TRUE)
        random_var <- var(decomp$random, na.rm = TRUE)
        total_var <- var(sleep_ts, na.rm = TRUE)
        
        list(
            trend_variance = trend_var,
            seasonal_variance = seasonal_var,
            random_variance = random_var,
            total_variance = total_var,
            seasonal_strength = seasonal_var / (seasonal_var + random_var)
        )
    } else {
        list(error = "Insufficient data for time series decomposition")
    }
    """
    
    try:
        result = cycle_analyzer.run_r_analysis(r_timeseries)
        if result['success']:
            ts_result = result['result']
            print("R Time Series Analysis Results:")
            if 'error' not in ts_result.names:
                print(f"Trend variance: {float(ts_result[0][0]):.4f}")
                print(f"Seasonal variance: {float(ts_result[1][0]):.4f}")
                print(f"Random variance: {float(ts_result[2][0]):.4f}")
                print(f"Total variance: {float(ts_result[3][0]):.4f}")
                print(f"Seasonal strength: {float(ts_result[4][0]):.4f}")
            else:
                print(f"Error: {ts_result[0][0]}")
        else:
            print(f"Time series analysis failed: {result['error']}")
    except Exception as e:
        print(f"R time series analysis failed: {e}")
else:
    print("R integration not available - skipping time series analysis")

## 6. Correlation Analysis with External Factors

In [ ]:
# Analyze correlations with external factors
external_factors = {
    'stress_level': sample_data['stress_level'],
    'exercise_duration': sample_data['exercise_duration'],
    'heart_rate': sample_data['heart_rate']
}

correlations = cycle_analyzer.correlate_with_external_factors(
    cycle_phases, 
    external_factors
)

print("Correlation with Cycle Phases:")
for factor, corr in correlations.items():
    print(f"{factor}: {corr:.4f}")

## 7. Visualization of R Analysis Results

In [ ]:
# Create comprehensive visualization
plot_data = sample_data.copy()
plot_data['cycle_phase'] = cycle_phases

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Sleep duration by cycle phase with confidence intervals
import seaborn as sns
sns.boxplot(data=plot_data, x='cycle_phase', y='sleep_duration', ax=axes[0, 0])
axes[0, 0].set_title('Sleep Duration by Cycle Phase')
axes[0, 0].set_ylabel('Hours')

# Heart rate by cycle phase
sns.boxplot(data=plot_data, x='cycle_phase', y='heart_rate', ax=axes[0, 1])
axes[0, 1].set_title('Heart Rate by Cycle Phase')
axes[0, 1].set_ylabel('BPM')

# Temperature by cycle phase
sns.boxplot(data=plot_data, x='cycle_phase', y='body_temperature', ax=axes[0, 2])
axes[0, 2].set_title('Body Temperature by Cycle Phase')
axes[0, 2].set_ylabel('°F')

# Sleep duration over time with trend
axes[1, 0].plot(plot_data['date'], plot_data['sleep_duration'], alpha=0.7)
axes[1, 0].set_title('Sleep Duration Over Time')
axes[1, 0].set_ylabel('Hours')
axes[1, 0].tick_params(axis='x', rotation=45)

# Correlation heatmap
corr_data = plot_data[['sleep_duration', 'sleep_efficiency', 'heart_rate', 
                      'body_temperature', 'stress_level', 'exercise_duration']].corr()
sns.heatmap(corr_data, annot=True, cmap='coolwarm', center=0, ax=axes[1, 1])
axes[1, 1].set_title('Correlation Matrix')

# Cycle day effect on sleep
cycle_sleep = plot_data.groupby('cycle_day')['sleep_duration'].mean()
axes[1, 2].plot(cycle_sleep.index, cycle_sleep.values, 'o-')
axes[1, 2].set_title('Average Sleep Duration by Cycle Day')
axes[1, 2].set_xlabel('Cycle Day')
axes[1, 2].set_ylabel('Hours')

plt.tight_layout()
plt.show()

## Conclusion

This notebook demonstrated advanced features of the menstrual_cycle_analysis package with R integration:

1. **R Statistical Analysis**: Linear regression and MANOVA using R
2. **Time Series Analysis**: Decomposition of sleep patterns using R
3. **Advanced Visualizations**: Comprehensive plots showing cycle effects
4. **Correlation Analysis**: Examining relationships with external factors

The R integration provides access to advanced statistical methods that complement the Python-based analyses, enabling more sophisticated research workflows for menstrual cycle physiology studies.